In [ ]:
# Import libraries. You may or may not use all of these.
!pip install -q git+https://github.com/tensorflow/docs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
  # %tensorflow_version only exists in Colab.
  %tensorflow_version 2.x
except Exception:
  pass
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

import tensorflow_docs as tfdocs
import tensorflow_docs.plots
import tensorflow_docs.modeling

In [ ]:
# Import data
!wget https://cdn.freecodecamp.org/project-data/health-costs/insurance.csv
dataset = pd.read_csv('insurance.csv')
dataset.tail()

In [ ]:
# Check the dataset
dataset.head()

In [ ]:
# Convert categorical columns into numerical values

dataset = pd.get_dummies(
    dataset,
    columns=['sex', 'smoker', 'region'],
    dtype=float
)

dataset.head()

In [ ]:
# Separate the features and the target value

labels = dataset.pop('expenses')

features = dataset

In [ ]:
# Normalize the numerical features

normalizer = layers.Normalization()

normalizer.adapt(np.array(features))

normalizer(features.iloc[:5])

In [ ]:
# Split the dataset

train_dataset = features.sample(frac=0.8, random_state=0)
test_dataset = features.drop(train_dataset.index)

train_labels = labels[train_dataset.index]
test_labels = labels[test_dataset.index]

print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))

In [ ]:
# Build the model

model = keras.Sequential([
    normalizer,
    layers.Dense(64, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mean_absolute_error',
    metrics=['mean_absolute_error', 'mean_squared_error']
)

model.summary()

In [ ]:
# Train the model

history = model.fit(
    train_dataset,
    train_labels,
    validation_split=0.2,
    verbose=0,
    epochs=200
)

In [ ]:
# Plot training progress

plt.plot(history.history['mean_absolute_error'])
plt.plot(history.history['val_mean_absolute_error'])

plt.xlabel('Epoch')
plt.ylabel('Mean Absolute Error')

plt.legend(['Training', 'Validation'])
plt.show()

In [ ]:
# RUN THIS CELL TO TEST YOUR MODEL. DO NOT MODIFY CONTENTS.
# Test model by checking how well the model generalizes using the test set.
loss, mae, mse = model.evaluate(test_dataset, test_labels, verbose=2)

print("Testing set Mean Abs Error: {:5.2f} expenses".format(mae))

if mae < 3500:
  print("You passed the challenge. Great job!")
else:
  print("The Mean Abs Error must be less than 3500. Keep trying.")

# Plot predictions.
test_predictions = model.predict(test_dataset).flatten()

a = plt.axes(aspect='equal')
plt.scatter(test_labels, test_predictions)
plt.xlabel('True values (expenses)')
plt.ylabel('Predictions (expenses)')
lims = [0, 50000]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims,lims)
